# 15 神经网络基础 - 从感知机到梯度下降

本教程是 Transformer 系列教程的第一篇，将从最基础的神经网络概念讲起，帮助你理解神经网络是如何工作的，以及它们是如何被训练的。

## 什么是神经网络？

神经网络（Neural Network）是一种受人脑神经元结构启发的机器学习模型。它的基本思想是：

1. **神经元（Neuron）**：接收输入，进行计算，产生输出
2. **权重（Weight）**：表示输入的重要性程度
3. **偏置（Bias）**：调整神经元的激活阈值
4. **激活函数（Activation Function）**：引入非线性，使网络能学习复杂的模式

让我们从最简单的**感知机（Perceptron）**开始理解。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import math

# 设置随机种子保证结果可复现
torch.manual_seed(42)
np.random.seed(42)

# 设置matplotlib中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("导入库成功！")

## 一、感知机（Perceptron）

感知机是最简单的神经网络单元，它只做一件事：

$$y = f(\mathbf{w} \cdot \mathbf{x} + b)$$

其中：
- $\mathbf{x} = [x_1, x_2, ..., x_n]$ 是输入向量
- $\mathbf{w} = [w_1, w_2, ..., w_n]$ 是权重向量
- $b$ 是偏置
- $f$ 是激活函数

### 数值示例

假设我们有一个感知机，输入 $\mathbf{x} = [2, 3]$，权重 $\mathbf{w} = [0.5, -0.2]$，偏置 $b = 0.1$：

$$z = \mathbf{w} \cdot \mathbf{x} + b = 0.5 \times 2 + (-0.2) \times 3 + 0.1 = 1.0 - 0.6 + 0.1 = 0.5$$

In [ ]:
# 感知机计算示例
x = torch.tensor([2.0, 3.0])
w = torch.tensor([0.5, -0.2])
b = torch.tensor(0.1)

# 计算 z = w·x + b
z = torch.dot(w, x) + b
print(f"输入 x = {x}")
print(f"权重 w = {w}")
print(f"偏置 b = {b}")
print(f"输出 z = w·x + b = {z}")

## 二、激活函数（Activation Functions）

激活函数是神经网络中**引入非线性**的关键。没有激活函数，无论多少层网络都等价于一个线性变换。

### 常见激活函数

| 激活函数 | 公式 | 特点 |
|---------|------|------|
| Sigmoid | $\sigma(x) = \frac{1}{1 + e^{-x}}$ | 输出范围 (0,1)，常用于二分类输出层 |
| Tanh | $\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$ | 输出范围 (-1,1)，零中心化 |
| ReLU | $\text{ReLU}(x) = \max(0, x)$ | 计算简单，最常用 |
| Leaky ReLU | $\text{LeakyReLU}(x) = \max(\alpha x, x)$ | 解决ReLU"死亡"问题 |
| GELU | $\text{GELU}(x) = x \cdot \Phi(x)$ | Transformer常用，平滑非线性 |

In [ ]:
# 定义各种激活函数
def sigmoid(x):
    return 1 / (1 + torch.exp(-x))

def tanh_func(x):
    return torch.tanh(x)

def relu(x):
    return torch.maximum(torch.tensor(0.0), x)

def leaky_relu(x, alpha=0.01):
    return torch.maximum(torch.tensor(alpha * x), x)

def gelu(x):
    # GELU近似: x * 0.5 * (1 + tanh(sqrt(2/pi) * (x + 0.044715 * x^3)))
    return 0.5 * x * (1 + torch.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * x**3)))

# 生成输入数据
x = torch.linspace(-5, 5, 200)

# 计算各激活函数的输出
y_sigmoid = sigmoid(x)
y_tanh = tanh_func(x)
y_relu = relu(x)
y_leaky_relu = leaky_relu(x)
y_gelu = gelu(x)

# 可视化激活函数
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Common Activation Functions', fontsize=16, fontweight='bold')

functions = [
    ('Sigmoid', y_sigmoid, '(0, 1)'),
    ('Tanh', y_tanh, '(-1, 1)'),
    ('ReLU', y_relu, '[0, +∞)'),
    ('Leaky ReLU', y_leaky_relu, '(-∞, +∞)'),
    ('GELU', y_gelu, '(-∞, +∞)'),
]

for idx, (name, y, range_desc) in enumerate(functions):
    row, col = idx // 3, idx % 3
    ax = axes[row, col]
    ax.plot(x.numpy(), y.numpy(), linewidth=2)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    ax.set_title(f'{name}\n输出范围: {range_desc}', fontsize=11)
    ax.set_xlabel('x')
    ax.set_ylabel('f(x)')
    ax.grid(True, alpha=0.3)

# 隐藏最后一个子图
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

print("各激活函数在 x=1 时的值：")
print(f"  Sigmoid(1)  = {sigmoid(torch.tensor(1.0)):.4f}")
print(f"  Tanh(1)     = {tanh_func(torch.tensor(1.0)):.4f}")
print(f"  ReLU(1)     = {relu(torch.tensor(1.0)):.4f}")
print(f"  LeakyReLU(1)= {leaky_relu(torch.tensor(1.0)):.4f}")
print(f"  GELU(1)     = {gelu(torch.tensor(1.0)):.4f}")

### 激活函数的导数（为什么重要？）

在训练神经网络时，我们需要通过**反向传播**计算梯度。激活函数的导数直接影响梯度如何流动：

- **Sigmoid导数**: $\sigma'(x) = \sigma(x)(1 - \sigma(x))$，最大值仅 0.25，多层连乘会导致**梯度消失**
- **Tanh导数**: $\tanh'(x) = 1 - \tanh^2(x)$，最大值 1，比Sigmoid好但仍可能梯度消失
- **ReLU导数**: 当 $x > 0$ 时导数为 1，当 $x \leq 0$ 时导数为 0，有效缓解梯度消失

In [ ]:
# 可视化激活函数的导数
def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

def tanh_derivative(x):
    return 1 - torch.tanh(x)**2

def relu_derivative(x):
    return (x > 0).float()

# 计算导数
y_sigmoid_d = sigmoid_derivative(x)
y_tanh_d = tanh_derivative(x)
y_relu_d = relu_derivative(x)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Derivatives of Activation Functions', fontsize=16, fontweight='bold')

axes[0].plot(x.numpy(), y_sigmoid_d.numpy(), 'r-', linewidth=2, label="σ'(x)")
axes[0].axhline(y=0.25, color='gray', linestyle='--', label='max=0.25')
axes[0].set_title('Sigmoid 导数\n(最大值仅0.25, 易梯度消失)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(x.numpy(), y_tanh_d.numpy(), 'b-', linewidth=2, label="tanh'(x)")
axes[1].axhline(y=1.0, color='gray', linestyle='--', label='max=1.0')
axes[1].set_title('Tanh 导数\n(最大值为1)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(x.numpy(), y_relu_d.numpy(), 'g-', linewidth=2, label="ReLU'(x)")
axes[2].set_title('ReLU 导数\n(正区间为1, 有效缓解梯度消失)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 三、损失函数（Loss Functions）

损失函数衡量模型预测值与真实值的差距。训练的目标就是**最小化损失函数**。

### 常见损失函数

1. **均方误差（MSE）**：用于回归任务
   $$L = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

2. **交叉熵损失（Cross-Entropy）**：用于分类任务
   $$L = -\sum_{i} y_i \log(\hat{y}_i)$$

3. **二元交叉熵（BCE）**：用于二分类
   $$L = -[y\log(\hat{y}) + (1-y)\log(1-\hat{y})]$$

In [ ]:
# 演示均方误差（MSE）损失
def mse_loss(predictions, targets):
    return torch.mean((predictions - targets)**2)

# 示例：模型预测 vs 真实值
targets = torch.tensor([1.0, 2.0, 3.0, 4.0])

# 三个不同的预测
pred_good = torch.tensor([1.1, 1.9, 3.1, 3.9])   # 接近真实值
pred_mid = torch.tensor([1.5, 1.5, 3.5, 3.5])   # 中等
pred_bad = torch.tensor([2.0, 0.5, 5.0, 2.0])   # 远离真实值

print("=== 均方误差（MSE）示例 ===")
print(f"好的预测: MSE = {mse_loss(pred_good, targets):.4f}")
print(f"中等预测: MSE = {mse_loss(pred_mid, targets):.4f}")
print(f"差的预测: MSE = {mse_loss(pred_bad, targets):.4f}")

# 可视化MSE损失曲面
w_range = np.linspace(-2, 4, 100)
b_range = np.linspace(-3, 3, 100)
W, B = np.meshgrid(w_range, b_range)

# 简单线性模型 y = wx + b，真实数据 x=1, y=2
x_data = np.array([1.0])
y_data = np.array([2.0])
Z = (W * x_data + B - y_data)**2

fig = plt.figure(figsize=(12, 5))

# 3D曲面图
ax1 = fig.add_subplot(121, projection='3d')
surf = ax1.plot_surface(W, B, Z, cmap='viridis', alpha=0.8)
ax1.set_xlabel('Weight w')
ax1.set_ylabel('Bias b')
ax1.set_zlabel('MSE Loss')
ax1.set_title('MSE Loss 3D Surface')
fig.colorbar(surf, ax=ax1, shrink=0.5)

# 等高线图
ax2 = fig.add_subplot(122)
contour = ax2.contourf(W, B, Z, levels=30, cmap='viridis')
ax2.plot([2.0], [0.0], 'r*', markersize=15, label='最优解 (w=2, b=0)')
ax2.set_xlabel('Weight w')
ax2.set_ylabel('Bias b')
ax2.set_title('MSE Loss Contour')
ax2.legend()
fig.colorbar(contour, ax=ax2)

plt.tight_layout()
plt.show()

## 四、梯度下降法（Gradient Descent）

梯度下降是训练神经网络的**核心算法**。它的基本思想很简单：

> **沿着损失函数下降最快的方向（梯度的反方向）更新参数**

### 更新公式

$$\theta := \theta - \eta \cdot \nabla_\theta L(\theta)$$

其中：
- $\theta$ 是模型参数（权重和偏置）
- $\eta$ 是**学习率（Learning Rate）**，控制每次更新的步长
- $\nabla_\theta L(\theta)$ 是损失函数对参数的梯度

### 学习率的影响

- **学习率太小**：收敛太慢，需要很多步才能到达最优
- **学习率太大**：可能越过最优解，甚至发散
- **学习率合适**：能快速、稳定地收敛到最优解

In [ ]:
# 一维梯度下降可视化
def f(x):
    """目标函数: f(x) = x^2 + 2x + 3 = (x+1)^2 + 2"""
    return x**2 + 2*x + 3

def df(x):
    """导数: f'(x) = 2x + 2"""
    return 2*x + 2

# 生成函数曲线
x_range = np.linspace(-4, 3, 200)
y_range = f(x_range)

# 三种不同学习率的梯度下降
learning_rates = [0.01, 0.1, 0.5]
colors_lr = ['blue', 'green', 'red']
labels_lr = ['η=0.01 (太小)', 'η=0.1 (合适)', 'η=0.5 (偏大)']

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x_range, y_range, 'k-', linewidth=2, label='f(x) = x² + 2x + 3')
ax.plot([-1], [f(-1)], 'r*', markersize=15, label='最小值 (-1, 2)')

for lr, color, label in zip(learning_rates, colors_lr, labels_lr):
    x = 3.0  # 初始值
    trajectory_x = [x]
    trajectory_y = [f(x)]
    
    for step in range(20):
        grad = df(x)
        x = x - lr * grad
        trajectory_x.append(x)
        trajectory_y.append(f(x))
    
    trajectory_x = np.array(trajectory_x)
    trajectory_y = np.array(trajectory_y)
    ax.plot(trajectory_x, trajectory_y, 'o-', color=color, label=label, 
            linewidth=1.5, markersize=4, alpha=0.7)

ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('f(x)', fontsize=12)
ax.set_title('Gradient Descent Trajectories with Different Learning Rates', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("梯度下降详细过程：")
print(f"{'步骤':<6} {'x':>10} {'f(x)':>10} {'梯度 f\'(x)':>12}")
print("-" * 45)
x = 3.0
lr = 0.1
for step in range(10):
    grad = df(x)
    print(f"{step:<6} {x:>10.4f} {f(x):>10.4f} {grad:>12.4f}")
    x = x - lr * grad

### 二维梯度下降

实际的神经网络有多个参数，让我们在二维空间观察梯度下降的行为。

In [ ]:
# 二维梯度下降
def loss_2d(w1, w2):
    """二元损失函数"""
    return 0.5 * w1**2 + 0.3 * w2**2 + 0.2 * w1 * w2

def grad_2d(w1, w2):
    """梯度"""
    dw1 = w1 + 0.2 * w2
    dw2 = 0.6 * w2 + 0.2 * w1
    return dw1, dw2

# 生成网格
w1_range = np.linspace(-3, 3, 100)
w2_range = np.linspace(-3, 3, 100)
W1, W2 = np.meshgrid(w1_range, w2_range)
Z = loss_2d(W1, W2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 3D图
ax1 = fig.add_subplot(121, projection='3d')
surf = ax1.plot_surface(W1, W2, Z, cmap='coolwarm', alpha=0.8)
ax1.set_xlabel('w1')
ax1.set_ylabel('w2')
ax1.set_zlabel('Loss')
ax1.set_title('2D Loss Function')
fig.colorbar(surf, ax=ax1, shrink=0.5)

# 等高线图 + 梯度下降轨迹
ax2 = fig.add_subplot(122)
contour = ax2.contourf(W1, W2, Z, levels=30, cmap='coolwarm')
fig.colorbar(contour, ax=ax2)

# 梯度下降轨迹
w1, w2 = 2.5, 2.5
lr = 0.3
path_w1 = [w1]
path_w2 = [w2]

for _ in range(30):
    dw1, dw2 = grad_2d(w1, w2)
    w1 = w1 - lr * dw1
    w2 = w2 - lr * dw2
    path_w1.append(w1)
    path_w2.append(w2)

ax2.plot(path_w1, path_w2, 'ko-', linewidth=2, markersize=4, label='梯度下降路径')
ax2.plot([0], [0], 'r*', markersize=15, label='最小值')
ax2.set_xlabel('w1')
ax2.set_ylabel('w2')
ax2.set_title('Gradient Descent Trajectory on Contour')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"初始点: (w1, w2) = (2.5, 2.5), Loss = {loss_2d(2.5, 2.5):.4f}")
print(f"最终点: (w1, w2) = ({w1:.4f}, {w2:.4f}), Loss = {loss_2d(w1, w2):.6f}")

## 五、用PyTorch训练一个简单的神经网络

现在我们把所有概念结合起来，用PyTorch训练一个简单的神经网络来拟合一个非线性函数。

### 训练流程

1. 准备数据
2. 定义网络结构
3. 定义损失函数和优化器
4. 训练循环：前向传播 → 计算损失 → 反向传播 → 更新参数
5. 可视化训练过程和结果

In [ ]:
# 生成训练数据：拟合 y = sin(x)
torch.manual_seed(42)

X_train = torch.linspace(-math.pi, math.pi, 200).unsqueeze(1)  # shape: (200, 1)
y_train = torch.sin(X_train)  # shape: (200, 1)

# 添加一些噪声
y_train = y_train + 0.1 * torch.randn_like(y_train)

# 可视化训练数据
plt.figure(figsize=(8, 5))
plt.scatter(X_train.numpy(), y_train.numpy(), s=10, alpha=0.5, label='带噪声的 sin(x)')
plt.plot(X_train.numpy(), torch.sin(X_train).numpy(), 'r-', linewidth=2, label='真实 sin(x)')
plt.xlabel('x')
plt.ylabel('y')
plt.title('训练数据: 拟合 sin(x)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"训练数据量: {len(X_train)}")
print(f"输入特征维度: {X_train.shape[1]}")
print(f"目标值维度: {y_train.shape[1]}")

In [ ]:
# 定义神经网络模型
class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        # 网络结构: 1 -> 32 -> 32 -> 1
        self.network = nn.Sequential(
            nn.Linear(1, 32),     # 输入层 -> 隐藏层1
            nn.ReLU(),            # ReLU激活
            nn.Linear(32, 32),    # 隐藏层1 -> 隐藏层2
            nn.ReLU(),            # ReLU激活
            nn.Linear(32, 1)      # 隐藏层2 -> 输出层
        )
    
    def forward(self, x):
        return self.network(x)

# 创建模型并查看结构
model = SimpleNN()
print(model)
print(f"\n总参数量: {sum(p.numel() for p in model.parameters())}")

# 查看初始参数
print("\n第一层权重 (初始化后):")
print(f"  形状: {model.network[0].weight.shape}")
print(f"  范围: [{model.network[0].weight.min().item():.4f}, {model.network[0].weight.max().item():.4f}]")

In [ ]:
# 训练模型
torch.manual_seed(42)
model = SimpleNN()

# 定义损失函数和优化器
criterion = nn.MSELoss()  # 均方误差
optimizer = optim.SGD(model.parameters(), lr=0.01)  # 随机梯度下降

# 训练参数
epochs = 500
loss_history = []

print("开始训练...")
print(f"{'Epoch':<8} {'Loss':>12} {'LR':>10}")
print("-" * 35)

for epoch in range(epochs):
    # 前向传播
    predictions = model(X_train)
    loss = criterion(predictions, y_train)
    
    # 反向传播
    optimizer.zero_grad()  # 清零梯度
    loss.backward()        # 计算梯度
    optimizer.step()       # 更新参数
    
    loss_history.append(loss.item())
    
    # 每50轮打印一次
    if (epoch + 1) % 50 == 0:
        print(f"{epoch+1:<8} {loss.item():>12.6f} {'0.01':>10}")

print(f"\n训练完成！最终损失: {loss_history[-1]:.6f}")

In [ ]:
# 可视化训练过程和结果
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. 损失曲线
axes[0].plot(loss_history, 'b-', linewidth=1.5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training Loss Curve')
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')  # 对数坐标

# 2. 拟合结果
model.eval()
with torch.no_grad():
    y_pred = model(X_train)

axes[1].scatter(X_train.numpy(), y_train.numpy(), s=10, alpha=0.3, label='Training Data')
axes[1].plot(X_train.numpy(), y_pred.numpy(), 'r-', linewidth=2, label='Model Prediction')
axes[1].plot(X_train.numpy(), torch.sin(X_train).numpy(), 'g--', linewidth=1.5, label='真实 sin(x)')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].set_title('Model Fitting Result')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. 权重分布
weights = []
for name, param in model.named_parameters():
    if 'weight' in name:
        weights.extend(param.detach().numpy().flatten())

axes[2].hist(weights, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
axes[2].set_xlabel('Weight Value')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Weight Distribution After Training')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 计算拟合优度
with torch.no_grad():
    y_pred = model(X_train)
    ss_res = torch.sum((y_train - y_pred)**2)
    ss_tot = torch.sum((y_train - y_train.mean())**2)
    r_squared = 1 - ss_res / ss_tot
    print(f"R² (拟合优度): {r_squared.item():.4f}")
    print(f"  (越接近1表示拟合越好)"

## 六、观察训练过程的动态变化

让我们保存训练过程中每个epoch的预测结果，观察模型是如何一步步学会拟合数据的。

In [ ]:
# 重新训练并记录每个epoch的预测
torch.manual_seed(42)
model = SimpleNN()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

epochs_to_save = [0, 10, 50, 100, 200, 499]
predictions_at_epochs = {}
loss_history2 = []

for epoch in range(500):
    predictions = model(X_train)
    loss = criterion(predictions, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    loss_history2.append(loss.item())
    
    if epoch in epochs_to_save:
        predictions_at_epochs[epoch] = predictions.detach().clone()

# 可视化不同阶段的预测
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Predictions at Different Training Stages', fontsize=16, fontweight='bold')
axes = axes.flatten()

for idx, epoch in enumerate(epochs_to_save):
    ax = axes[idx]
    ax.scatter(X_train.numpy(), y_train.numpy(), s=5, alpha=0.3, label='数据')
    ax.plot(X_train.numpy(), predictions_at_epochs[epoch].numpy(), 'r-', 
            linewidth=2, label=f'Epoch {epoch}')
    ax.plot(X_train.numpy(), torch.sin(X_train).numpy(), 'g--', linewidth=1, alpha=0.5)
    ax.set_title(f'Epoch {epoch}\nLoss = {loss_history2[epoch]:.4f}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 七、不同学习率的对比实验

学习率是神经网络训练中最重要的超参数。让我们通过实验来理解不同学习率的影响。

In [ ]:
# 比较不同学习率的训练效果
learning_rates = [0.001, 0.01, 0.05, 0.1]
lr_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Training Comparison with Different Learning Rates', fontsize=16, fontweight='bold')
axes = axes.flatten()

for idx, lr in enumerate(learning_rates):
    torch.manual_seed(42)
    model_lr = SimpleNN()
    criterion_lr = nn.MSELoss()
    optimizer_lr = optim.SGD(model_lr.parameters(), lr=lr)
    
    loss_hist = []
    for epoch in range(500):
        pred = model_lr(X_train)
        loss = criterion_lr(pred, y_train)
        optimizer_lr.zero_grad()
        loss.backward()
        optimizer_lr.step()
        loss_hist.append(loss.item())
    
    ax = axes[idx]
    ax.plot(loss_hist, color=lr_colors[idx], linewidth=2, label=f'lr={lr}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title(f'学习率 η = {lr}\n最终损失 = {loss_hist[-1]:.6f}')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    
    # 在同一张图上画拟合结果
    model_lr.eval()
    with torch.no_grad():
        y_pred_lr = model_lr(X_train)
    
    ax.plot([], [], color=lr_colors[idx], linewidth=3)  # 占位

plt.tight_layout()
plt.show()

# 汇总比较
print(f"{'学习率':<10} {'最终Loss':>12} {'收敛速度':>10}")
print("-" * 35)
for lr in learning_rates:
    torch.manual_seed(42)
    model_lr = SimpleNN()
    optimizer_lr = optim.SGD(model_lr.parameters(), lr=lr)
    
    for epoch in range(500):
        pred = model_lr(X_train)
        loss = criterion(pred, y_train)
        optimizer_lr.zero_grad()
        loss.backward()
        optimizer_lr.step()
    
    print(f"{lr:<12} {loss.item():>12.6f} {'-'*10:>10}")

## 八、本章总结

### 核心概念回顾

1. **感知机**：神经网络的基本单元，计算 $\mathbf{w} \cdot \mathbf{x} + b$

2. **激活函数**：引入非线性，使网络能学习复杂模式
   - Sigmoid/Tanh：容易梯度消失，现代网络较少用
   - ReLU：简单高效，最常用
   - GELU：平滑非线性，Transformer常用

3. **损失函数**：衡量预测与真实值的差距
   - MSE：回归任务
   - 交叉熵：分类任务

4. **梯度下降**：沿梯度反方向更新参数
   - 学习率是关键超参数
   - 太大可能发散，太小收敛慢

5. **训练流程**：
   - 前向传播 → 计算损失 → 反向传播 → 更新参数

### 下一篇预告

在下一篇教程中，我们将深入学习**优化算法**：
- SGD（随机梯度下降）的变种
- 动量法（Momentum）
- Adam优化器（最流行的自适应优化器）
- 不同优化算法的对比实验